# 07｜微分應用：極值、均值定理、最佳化、羅必達

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/calculus-for-iih/blob/main/notebooks/07_applications.ipynb)

臺北醫學大學 智慧醫療跨領域　微積分（1151）　**自由練習，不計分、不用繳交**

對應課堂投影片：**Calculus07 微分應用**

**怎麼用**
1. 按上方選單「執行階段 → 全部執行」，從上往下看圖和數字。
2. 看到 **✏️ 動手** 的格子，就自己改改看。填空沒填也不會出錯，程式會提醒你「還沒填」。
3. 卡住就點開「💡 提示」，真的不行再看「參考答案」。

不需要先會寫 Python，每一段程式前面都有中文說明。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def d(f, a, h=1e-5):
    "中央差分：用前後兩點估計導數（老師提供）。"
    return (f(a + h) - f(a - h)) / (2 * h)

def d2(f, a, h=1e-4):
    "二階導數的數值估計（老師提供）。"
    return (f(a + h) - 2 * f(a) + f(a - h)) / h**2

def check(ok, hint):
    """檢查答案：對了顯示 ✓，不對顯示提示。"""
    print("✓ 正確！" if ok else "✗ 還沒對，" + hint)

## 1. f′ 的正負：上坡還是下坡？

$f(x)=x^3-3x$。把 $f'>0$（遞增）的地方塗綠色、$f'<0$（遞減）的地方塗紅色，再找 $f'$ 變號的位置。

In [ ]:
f = lambda x: x**3 - 3 * x
x = np.linspace(-2.3, 2.3, 461)
slope = d(f, x)

plt.figure(figsize=(7, 3.8))
plt.plot(x, f(x), color="#0B2E33", lw=3)
plt.fill_between(x, -5, 5, where=slope > 0, color="#00A896", alpha=0.12, label="f' > 0 (increasing)")
plt.fill_between(x, -5, 5, where=slope < 0, color="#D9534F", alpha=0.12, label="f' < 0 (decreasing)")
plt.ylim(-4, 4); plt.xlabel("x"); plt.grid(alpha=0.3); plt.legend(loc="lower right"); plt.show()

# 找 f′ 變號的地方：相鄰兩點的正負號不同
for i in np.where(np.diff(np.sign(slope)) != 0)[0]:
    kind = "局部極大（正 → 負）" if slope[i] > 0 else "局部極小（負 → 正）"
    print(f"x ≈ {x[i + 1]:.2f}：{kind}，f ≈ {f(x[i + 1]):.2f}")

## 2. ✏️ 動手 1：f(x) = x³ − 12x 的局部極大

把上面的 `f` 換成 $x^3-12x$ 也可以。請先手算：$f'(x)=3x^2-12$，局部極大在 $x=$ 多少？

In [ ]:
answer_1 = None      # ✏️ 填空：局部極大的 x

# ---- 自動檢查（不用改）----
g = lambda x: x**3 - 12 * x
if answer_1 is None:
    print("還沒填，令 3x² − 12 = 0，再看哪一個是正 → 負。")
else:
    check(abs(d(g, answer_1)) < 1e-3 and d2(g, answer_1) < 0, "f′ = 0 的兩個點中，f″ < 0 的那個才是極大。")

## 3. f′ = 0 不一定是極值

$x^3$ 在 0 的導數是 0，但 $f'$ 在兩側都是正的，沒有變號。

In [ ]:
for v in [-0.1, -0.01, 0, 0.01, 0.1]:
    print(f"x = {v:>5}：x³ = {v**3:>9.6f}，f′ ≈ {d(lambda t: t**3, v):.4f}")

函數值一路在增加，所以 $x=0$ 不是極值。**$f'=0$ 只是「候選人」。**

## 4. 口服藥：濃度何時最高？何時下降最快？

$C(t)=20t\,e^{-t/2}$（mg/L，t 為小時，示意模型）。
- $C'=0$ 的時刻：濃度最高（Tmax、Cmax）。
- $C''=0$ 的時刻（反曲點）：濃度下降最快。

In [ ]:
C = lambda t: 20 * t * np.exp(-t / 2)
t = np.linspace(0, 12, 1201)

fig, ax = plt.subplots(3, 1, figsize=(7, 6.5), sharex=True)
ax[0].plot(t, C(t), color="#0B2E33", lw=3); ax[0].set_ylabel("C (mg/L)")
ax[1].plot(t, d(C, t), color="#028090", lw=3); ax[1].set_ylabel("C'")
ax[2].plot(t, d2(C, t), color="#D9534F", lw=3); ax[2].set_ylabel("C''"); ax[2].set_xlabel("t (hours)")
for a in ax: a.axhline(0, color="gray", lw=1); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

i_max = C(t).argmax()
i_fast = d(C, t).argmin()
print(f"Tmax ≈ {t[i_max]:.2f} 小時，Cmax ≈ {C(t[i_max]):.2f} mg/L（公式：40/e = {40 / np.e:.2f}）")
print(f"下降最快在 t ≈ {t[i_fast]:.2f} 小時，C′ ≈ {d(C, t[i_fast]):.2f} mg/L/h")

## 5. ✏️ 動手 2：另一種藥

$C(t)=30t\,e^{-t/4}$（示意）。微分得 $C'(t)=30e^{-t/4}(1-t/4)$。什麼時候濃度最高？先手算，再改上面的程式驗證。

In [ ]:
answer_2 = None      # ✏️ 填空：Tmax（小時）

# ---- 自動檢查（不用改）----
if answer_2 is None:
    print("還沒填，令 1 − t/4 = 0。")
else:
    C2 = lambda t: 30 * t * np.exp(-t / 4)
    check(abs(d(C2, answer_2)) < 1e-3, "C′ = 0 的時刻；e 的次方永遠是正的，只看括號。")

## 6. 從虛擬抽血數據估 Tmax

真實世界拿不到公式，只有每隔一段時間的抽血值，而且有量測誤差。下面產生**虛擬**數據，用相鄰兩點的差（差商）看濃度從「上升」變成「下降」的地方。

In [ ]:
rng = np.random.default_rng(7)
t_obs = np.arange(0, 12.5, 0.5)
C_obs = C(t_obs) + rng.normal(0, 0.3, t_obs.size)

rate = np.diff(C_obs) / np.diff(t_obs)          # 每段的平均變化率
t_mid = (t_obs[:-1] + t_obs[1:]) / 2

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(t_obs, C_obs, "o", color="#D9534F", label="virtual data")
ax[0].plot(t, C(t), color="#BFDEDE", lw=3, label="model")
ax[0].set_xlabel("t (hours)"); ax[0].set_ylabel("C (mg/L)"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].bar(t_mid, rate, width=0.4, color=np.where(rate > 0, "#00A896", "#D9534F"))
ax[1].axhline(0, color="gray"); ax[1].set_xlabel("t (hours)"); ax[1].set_ylabel("rate (mg/L/h)"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("數據最高點出現在 t =", t_obs[C_obs.argmax()], "小時（模型的 Tmax 是 2 小時）")

雜訊可能讓單一最高點偏前或偏後，後段也會冒出幾根「假的」綠色柱子（濃度其實在下降）。所以要看變化率**整體**由正轉負的位置，不要被單一個點騙了。

## 7. ✏️ 動手 3：閉區間的最大值

$f(x)=x^2-4x+1$ 在 $[0,5]$ 上。候選人是臨界點 $x=2$ 和兩個端點。電腦可以直接在很密的格點上找最大值，請和手算比對。

In [ ]:
h = lambda x: x**2 - 4 * x + 1
xs = np.linspace(0, 5, 5001)
print("候選值：f(0) =", h(0), "，f(2) =", h(2), "，f(5) =", h(5))
print("格點搜尋：最大", round(h(xs).max(), 3), "在 x =", xs[h(xs).argmax()], "；最小", round(h(xs).min(), 3), "在 x =", xs[h(xs).argmin()])

answer_3 = None      # ✏️ 填空：f 在 [0, 5] 上的最大值

# ---- 自動檢查（不用改）----
if answer_3 is None:
    print("還沒填，三個候選值比大小，別忘了端點。")
else:
    check(abs(answer_3 - 6) < 1e-6, "最大值在端點 x = 5。")

## 8. 均值定理（MVT）：區間測速

一段 10 km 的路，開了 6 分鐘，平均 100 km/h。下面是一條**虛擬**的車速紀錄（忽快忽慢）。MVT 保證：至少有一個瞬間，車速剛好等於平均速度。

In [ ]:
T_total = 0.1                                          # 6 分鐘 = 0.1 小時
tt = np.linspace(0, T_total, 601)
speed = 100 + 18 * np.sin(2 * np.pi * tt / T_total * 1.5) * np.exp(-8 * tt)   # km/h（虛擬）
area = lambda y, x: np.sum((y[1:] + y[:-1]) / 2 * np.diff(x))                 # 梯形法算距離
speed = speed * 10 / area(speed, tt)                                            # 調整成總距離剛好 10 km
dist = area(speed, tt)
avg = dist / T_total

plt.figure(figsize=(7, 3.5))
plt.plot(tt * 60, speed, color="#0B2E33", lw=3, label="speed")
plt.axhline(avg, color="#D9534F", ls="--", label=f"average {avg:.1f} km/h")
plt.xlabel("minutes"); plt.ylabel("km/h"); plt.grid(alpha=0.3); plt.legend(); plt.show()

hits = tt[np.where(np.diff(np.sign(speed - avg)) != 0)[0]] * 60
print(f"總距離 {dist:.2f} km，平均 {avg:.1f} km/h")
print("車速剛好等於平均速度的時刻（分鐘）：", np.round(hits, 2))

不管怎麼開，曲線一定會穿過平均速度那條線。這就是區間測速不用在半路抓到你的原因（情境為示意）。

## 9. 最佳化：紙盒裝最多

20 × 30 cm 的紙板四角各剪掉邊長 x 的正方形：$V(x)=x(20-2x)(30-2x)$，$0<x<10$。

In [ ]:
V = lambda x: x * (20 - 2 * x) * (30 - 2 * x)
x_cut = 3       # ← 改這裡，看容積怎麼變

xs = np.linspace(0, 10, 1001)
plt.figure(figsize=(6.5, 3.5))
plt.plot(xs, V(xs), color="#0B2E33", lw=3)
plt.plot(x_cut, V(x_cut), "o", color="#E8A33D", ms=10)
plt.xlabel("x (cm)"); plt.ylabel("V (cm³)"); plt.grid(alpha=0.3); plt.show()
print(f"x = {x_cut} cm：V = {V(x_cut):.1f} cm³，V′ ≈ {d(V, x_cut):.1f}")

## 10. ✏️ 動手 4：最佳的 x

微分 $V'(x)=12x^2-200x+600=0$，用公式解。也可以一直改上面的 `x_cut`，直到 $V'$ 接近 0。

In [ ]:
answer_4 = None      # ✏️ 填空：容積最大的 x（小數兩位即可）

# ---- 自動檢查（不用改）----
if answer_4 is None:
    print("還沒填，用 np.roots([12, -200, 600]) 解方程式，選在 0 到 10 之間的那一個。")
else:
    check(abs(answer_4 - 3.924) < 0.02, "另一個根 12.74 不在 0 < x < 10 之內，要丟掉。")

## 11. 劑量多少最划算？

療效 $E(d)=\dfrac{100d}{d+10}$，副作用 $S(d)=0.5d$，淨效益 $B=E-S$（示意，非特定藥品）。最佳劑量時，多給 1 mg 增加的療效剛好等於增加的副作用：$E'(d)=S'(d)$。

In [ ]:
E = lambda dose: 100 * dose / (dose + 10)
S = lambda dose: 0.5 * dose
B = lambda dose: E(dose) - S(dose)
ds = np.linspace(0, 100, 10001)
best = ds[B(ds).argmax()]

plt.figure(figsize=(6.5, 3.5))
plt.plot(ds, E(ds), "--", color="#028090", label="E (benefit)")
plt.plot(ds, S(ds), "--", color="#D9534F", label="S (side effect)")
plt.plot(ds, B(ds), color="#0B2E33", lw=3, label="B = E - S")
plt.axvline(best, color="#E8A33D"); plt.xlabel("dose (mg)"); plt.grid(alpha=0.3); plt.legend(); plt.show()

print(f"最佳劑量 ≈ {best:.1f} mg，B ≈ {B(best):.1f}（公式 20√5 − 10 = {20 * np.sqrt(5) - 10:.1f}）")
print(f"此時 E′ ≈ {d(E, best):.3f}，S′ = 0.5")

## 12. 連結 AI：梯度下降與學習率

$L(w)=(w-3)^2+1$。每一步 $w \leftarrow w-\eta\,L'(w)$。改 `eta` 試試：0.25、0.9、1.1，會發生什麼事？

In [ ]:
L = lambda w: (w - 3)**2 + 1
eta = 0.25      # ← 改這裡：學習率

w = 0.0
path = [w]
for step in range(10):
    w = w - eta * d(L, w)
    path.append(w)
path = np.array(path)

ws = np.linspace(-1, 7, 200)
plt.figure(figsize=(6.5, 3.5))
plt.plot(ws, L(ws), color="#0B2E33", lw=3)
plt.plot(path, L(path), "o-", color="#D9534F")
plt.ylim(0, 20); plt.xlabel("w"); plt.ylabel("L(w)"); plt.grid(alpha=0.3); plt.show()
print("w 的軌跡：", np.round(path, 3))

學習率太大（例如 1.1），每一步都跨過谷底而且越跳越遠：AI 訓練「發散」了。

## 13. 羅必達法則：用數值看極限

$\displaystyle\lim_{x\to0}\frac{e^x-1}{x}=1$、$\displaystyle\lim_{x\to0}\frac{1-\cos x}{x^2}=\frac12$。讓 x 越來越接近 0，看看數值。

In [ ]:
print("     x        (eˣ−1)/x     (1−cos x)/x²")
for x0 in [0.1, 0.01, 0.001, 0.0001]:
    print(f"{x0:>8}   {(np.exp(x0) - 1) / x0:.6f}     {(1 - np.cos(x0)) / x0**2:.6f}")

import sympy as sp
X = sp.symbols("x")
print()
print("SymPy：", sp.limit((sp.exp(X) - 1) / X, X, 0), "，", sp.limit((1 - sp.cos(X)) / X**2, X, 0),
      "，x/eˣ 在 x→∞：", sp.limit(X / sp.exp(X), X, sp.oo))

## 14. 用程式檢查 AI 的答案

> AI 說：$\displaystyle\lim_{x\to0}\frac{x+1}{x+2}$ 用羅必達法則，分子分母各自微分得 $\frac11=1$。

In [ ]:
q = lambda x: (x + 1) / (x + 2)
for x0 in [0.1, 0.001, -0.001]:
    print(f"x = {x0:>6}：{q(x0):.5f}")
print("直接代入 x = 0：", q(0))

數值接近 0.5，不是 1。代入 $x=0$ 根本不是 $0/0$，**不能用羅必達**，直接代入就好。用任何工具之前，先檢查它的條件。

## 小結

- $f'$ 變號的地方才是極值；$f'=0$ 只是候選人。閉區間別忘了端點。
- $f''$ 決定凹向；反曲點是變化最快（或最慢）的瞬間。
- MVT：平均變化率，中間某一刻一定剛好踩到。
- 最佳化：寫出只有一個變數的函數 → 微分 → 回到原問題。
- 羅必達只用在 $0/0$、$\infty/\infty$。

本週內容會考在期末考。

---
*本 notebook 為教學練習，數值與情境皆為示意，資料為程式產生的虛擬資料，非真實病人資料，也不是用藥建議。*